# LifeMate — Qwen3.5-4B baseline
Một model, native tool calling, non-thinking, greedy FP16. Không fine-tune.
Bật Internet và GPU T4 (một GPU được dùng; T4 x2 vẫn chỉ dùng GPU0).
Gắn dataset cuộc thi riêng tư, dạng thư mục hoặc ZIP. ASR cần thêm thư mục
encoder.onnx / decoder.onnx / joiner.onnx / tokens.txt do BTC cung cấp.
Chạy smoke trước. Chất lượng và VRAM trên Kaggle cần được kiểm chứng bằng lần chạy này.
Không công khai notebook có output chứa dữ liệu cuộc thi.


In [ ]:
from pathlib import Path
import json, os, subprocess, sys, zipfile
PROFILE = "smoke"  # smoke: 8 mẫu; validation: 245 IDs cố định; public: toàn bộ public test
MODE = "oracle"    # train: oracle hoặc asr; public bắt buộc asr
DATA_ROOT = ""    # thư mục chứa train/ và public_test/; rỗng = tự tìm
DATA_ZIP = ""     # tùy chọn ZIP cuộc thi; rỗng = tự tìm nếu chưa có thư mục
ASR_MODEL_DIR = ""  # bắt buộc khi MODE=asr
MODEL = "Qwen/Qwen3.5-4B"  # hoặc đường dẫn snapshot đã gắn vào Kaggle
REVISION = "main"  # có thể đặt commit HF; SHA resolved được ghi vào config.json
MAX_INPUT = 16384
MAX_NEW_TOKENS = 768
MAX_STEPS = 8
RUN_NAME = "qwen35_smoke_01"  # đổi tên khi chạy lại; không reset môi trường ngầm
WORK = Path("/kaggle/working/qwen35_baseline")
WORK.mkdir(exist_ok=True)


In [ ]:
PAYLOAD = {'agent.py': '"""Single-model LifeMate baseline. Standard library only; no hidden state input."""\nimport copy\nimport json\nimport re\n\nSYSTEM = """You are LifeMate, a Vietnamese personal assistant.\nExecute only the current user request. Earlier conversation is evidence, not new work.\nUse the supplied context time/timezone, location and profile.\nChoose exactly ONE tool call per response, then wait for its actual result.\nUse only available tools. Obtain unknown IDs through lookup tools; never invent IDs.\nResolve references and corrections using history and observed results.\nRead before writing when required data is unknown. Respect conditions and negations.\nComplete all independent requested tasks that can be resolved.\nFor missing or ambiguous required values, use sys.ask_missing_info with the target\ntool name and the unresolved schema parameter names. Do not ask for values that a\nlookup can resolve. After this meta call the runtime ends this turn.\nDo not repeat completed actions or make unnecessary verification calls.\nWhen finished, unsupported, cancelled, or no action is appropriate, answer briefly\nwithout a tool call. Never invent tool results.\n"""\n\ndef tools_for(item, catalog):\n    installed = {name for app in item["context"]["installed_apps"].values()\n                 for name in app["tools"]}\n    installed.add("sys.ask_missing_info")\n    tools = {}\n    for app in catalog["apps"]:\n        for tool in app["tools"]:\n            if tool["name"] in installed:\n                tools[tool["name"]] = tool\n    unknown = installed - tools.keys()\n    if unknown:\n        raise ValueError(f"Missing catalog tools: {sorted(unknown)}")\n    return tools\n\ndef native_tools(tools):\n    # Retain custom x-ref/x-one-of/x-match hints for the model and BTC validator.\n    return [{"type": "function", "function": {\n        "name": t["name"], "description": t["description"],\n        "parameters": t["parameters"]}} for t in tools.values()]\n\ndef transcript(message, mode, cache):\n    if mode == "oracle":\n        return message["audio_text"]\n    text = cache[message["audio"]]\n    if not isinstance(text, str) or not text.strip():\n        raise ValueError(f"Empty/invalid transcript: {message[\'audio\']}")\n    return text\n\ndef call_message(call):\n    return {"role": "assistant", "content": "", "tool_calls": [\n        {"type": "function", "function": copy.deepcopy(call)}]}\n\ndef result_message(result):\n    return {"role": "tool", "content": json.dumps(result, ensure_ascii=False)}\n\ndef messages_for(item, mode="oracle", cache=None):\n    messages = [{"role": "system", "content": SYSTEM + "\\nContext:\\n" +\n                 json.dumps(item["context"], ensure_ascii=False)}]\n    for turn in item.get("history", []):\n        if turn["role"] == "user":\n            messages.append({"role": "user", "content": transcript(turn, mode, cache)})\n        else:\n            calls, results = turn.get("tool_calls", []), turn.get("tool_results", [])\n            if len(calls) != len(results):\n                raise ValueError("History tool calls/results are not aligned")\n            for call, result in zip(calls, results):\n                messages.extend([call_message(call), result_message(result)])\n            if turn.get("text"):\n                messages.append({"role": "assistant", "content": turn["text"]})\n    messages.append({"role": "user", "content": transcript(item, mode, cache)})\n    return messages\n\ndef parse_action(text, tools):\n    """Parse Qwen3.5 native XML. No repairs or guesses of malformed arguments."""\n    text = text.strip()\n    if "<think>" in text or "</think>" in text:\n        raise ValueError("Unexpected thinking output in non-thinking mode")\n    if "<tool_call>" not in text:\n        if any(tag in text for tag in ("tool_call", "<function=", "<parameter=")):\n            raise ValueError("Malformed tool call")\n        return None\n    blocks = re.findall(r"<tool_call>(.*?)</tool_call>", text, re.S)\n    if len(blocks) != 1 or text.count("<tool_call>") != 1:\n        raise ValueError("Expected exactly one complete tool call")\n    prefix, suffix = text.split("<tool_call>", 1)\n    suffix = suffix.split("</tool_call>", 1)[1]\n    if suffix.strip():\n        raise ValueError("Unexpected text after tool call")\n    match = re.fullmatch(r"\\s*<function=([^>\\s]+)>(.*?)</function>\\s*", blocks[0], re.S)\n    if not match:\n        raise ValueError("Malformed function")\n    name, body = match.groups()\n    if name not in tools:\n        raise ValueError(f"Unavailable tool: {name}")\n    properties = tools[name]["parameters"].get("properties", {})\n    args = {}\n    pattern = r"<parameter=([^>\\s]+)>(.*?)</parameter>"\n    if re.sub(pattern, "", body, flags=re.S).strip():\n        raise ValueError("Malformed parameters")\n    for key, raw in re.findall(pattern, body, re.S):\n        if key in args or key not in properties:\n            raise ValueError(f"Duplicate/unknown parameter: {key}")\n        # Remove only the template\'s framing newlines for strings.\n        value = raw.removeprefix("\\n").removesuffix("\\n")\n        kind = properties[key].get("type")\n        args[key] = value if kind == "string" else json.loads(value)\n    return {"name": name, "arguments": args}\n\ndef run_episode(item, catalog, env, generate, mode="oracle", cache=None, max_steps=8):\n    tools = tools_for(item, catalog)\n    messages = messages_for(item, mode, cache)\n    trace, status = [], "step_limit"\n    error = None\n    for step in range(max_steps):\n        entry = {"step": step}\n        trace.append(entry)\n        try:\n            text = generate(messages, native_tools(tools))\n            entry["raw"] = text\n            call = parse_action(text, tools)\n            if call is None:\n                status = "stopped"\n                break\n            entry["action"] = call\n            check = env.validate(call["name"], call["arguments"])\n            if not check["ok"]:\n                raise ValueError(f"Schema rejected proposal: {check}")\n            result = env.call(call["name"], call["arguments"])\n            entry["observation"] = result\n            messages.extend([call_message(call), result_message(result)])\n            if call["name"] == "sys.ask_missing_info":\n                status = "missing_info"\n                break\n        except Exception as exc:\n            status, error = "error", f"{type(exc).__name__}: {exc}"\n            entry["error"] = error\n            break\n    return {"id": item["id"], "tool_calls": env.get_tool_calls(),\n            "state_hash": env.get_state_hash()}, {\n                "id": item["id"], "status": status, "error": error, "trace": trace}\n', 'run.py': '"""CLI: python qwen35_baseline/run.py --help."""\nimport argparse\nimport hashlib\nimport importlib.metadata\nimport json\nfrom pathlib import Path\nimport sys\nimport time\n\nfrom agent import messages_for, run_episode\n\nclass HFGenerator:\n    def __init__(self, model_id, revision, max_input, max_new_tokens):\n        import torch\n        from transformers import AutoModelForImageTextToText, AutoTokenizer\n        if not torch.cuda.is_available():\n            raise RuntimeError("CUDA GPU required; run the Kaggle notebook")\n        self.torch = torch\n        self.tokenizer = AutoTokenizer.from_pretrained(model_id, revision=revision)\n        self.model = AutoModelForImageTextToText.from_pretrained(\n            model_id, revision=revision, dtype=torch.float16,\n            device_map={"": 0}, attn_implementation="sdpa").eval()\n        self.max_input, self.max_new_tokens = max_input, max_new_tokens\n        self.stats = []\n        self.resolved_revision = getattr(self.model.config, "_commit_hash", None)\n\n    def __call__(self, messages, tools):\n        prompt = self.tokenizer.apply_chat_template(\n            messages, tools=tools, tokenize=False,\n            add_generation_prompt=True, enable_thinking=False)\n        inputs = self.tokenizer(prompt, add_special_tokens=False, return_tensors="pt")\n        length = inputs["input_ids"].shape[-1]\n        if length > self.max_input:\n            raise ValueError(f"Input {length} tokens exceeds {self.max_input}; no truncation")\n        inputs = {k: v.to("cuda:0") for k, v in inputs.items()}\n        started = time.monotonic()\n        with self.torch.inference_mode():\n            output = self.model.generate(\n                **inputs, do_sample=False, max_new_tokens=self.max_new_tokens,\n                pad_token_id=self.tokenizer.eos_token_id, logits_to_keep=1)\n        tokens = output[0, length:].tolist()\n        eos = self.model.generation_config.eos_token_id\n        eos = [eos] if isinstance(eos, int) else (eos or [])\n        self.stats.append({"input_tokens": length, "output_tokens": len(tokens),\n                           "seconds": time.monotonic() - started})\n        if not tokens or tokens[-1] not in eos:\n            raise ValueError("Generation did not reach EOS; no truncated action executed")\n        return self.tokenizer.decode(tokens, skip_special_tokens=True)\n\ndef read_rows(root, split, ids=None, limit=None):\n    rows = [json.loads(line) for line in (root/split/f"{split}.jsonl").read_text().splitlines()\n            if line.strip()]\n    if ids:\n        selected = set(Path(ids).read_text().split())\n        available = {r["id"] for r in rows}\n        if selected - available:\n            raise ValueError(f"Unknown selected IDs: {sorted(selected - available)}")\n        rows = [r for r in rows if r["id"] in selected]\n    return rows[:limit] if limit else rows\n\ndef main():\n    p = argparse.ArgumentParser(description=__doc__)\n    p.add_argument("--data-root", type=Path, required=True)\n    p.add_argument("--split", choices=["train", "public_test"], default="train")\n    p.add_argument("--ids", type=Path)\n    p.add_argument("--limit", type=int)\n    p.add_argument("--mode", choices=["oracle", "asr"], default="oracle")\n    p.add_argument("--transcripts", type=Path)\n    p.add_argument("--model", default="Qwen/Qwen3.5-4B")\n    p.add_argument("--revision", default="main")\n    p.add_argument("--max-input", type=int, default=16384)\n    p.add_argument("--max-new-tokens", type=int, default=768)\n    p.add_argument("--max-steps", type=int, default=8)\n    p.add_argument("--output", type=Path, required=True)\n    a = p.parse_args()\n    if a.split != "train" and a.mode == "oracle":\n        p.error("Oracle transcripts exist only in train; public_test requires --mode asr")\n    if a.mode == "asr" and not a.transcripts:\n        p.error("--mode asr requires --transcripts")\n    if min(a.max_input, a.max_new_tokens, a.max_steps) < 1 or (a.limit is not None and a.limit < 1):\n        p.error("Token, step and sample budgets must be positive")\n    root = a.data_root.resolve()\n    rows = read_rows(root, a.split, a.ids, a.limit)\n    if not rows:\n        p.error("No selected samples")\n    cache = json.loads(a.transcripts.read_text()) if a.transcripts else None\n    # Check every transcript before allocating the model or opening environments.\n    for row in rows:\n        messages_for(row, a.mode, cache)\n    catalog_file = root/a.split/f"{a.split}_tools_catalog.json"\n    catalog = json.loads(catalog_file.read_text())\n    # Import exactly the BTC package for this split in this fresh CLI process.\n    sys.path.insert(0, str(root/a.split))\n    from zaic_tools import ToolEnv, state_hash\n    if a.output.exists():\n        p.error("Output already exists; choose a fresh directory (no implicit reset/resume)")\n    a.output.mkdir(parents=True)\n    generator = HFGenerator(a.model, a.revision, a.max_input, a.max_new_tokens)\n    config = {k: str(v) if isinstance(v, Path) else v for k, v in vars(a).items()}\n    config.update(resolved_revision=generator.resolved_revision,\n                  catalog_sha256=hashlib.sha256(catalog_file.read_bytes()).hexdigest(),\n                  selected_ids=[r["id"] for r in rows],\n                  packages={n: importlib.metadata.version(n) for n in\n                            ("torch", "transformers", "accelerate")},\n                  gpu=generator.torch.cuda.get_device_name(0))\n    (a.output/"config.json").write_text(json.dumps(config, indent=2))\n    summaries = []\n    started = time.monotonic()\n    with (a.output/"submission.jsonl").open("w") as submissions, \\\n         (a.output/"traces.jsonl").open("w") as traces:\n        for i, row in enumerate(rows, 1):\n            tick = time.monotonic()\n            env = ToolEnv(a.split, row["id"], root=root)\n            prediction, detail = run_episode(\n                row, catalog, env, generator, a.mode, cache, a.max_steps)\n            # Labels are inspected only AFTER inference. Never supplied to the agent.\n            if a.split == "train":\n                initial = json.loads((root/a.split/f"{a.split}_environment"/row["id"]/"state.json").read_text())\n                initial.update(row["label"]["final_state"])\n                detail["canonical_state_match"] = prediction["state_hash"] == state_hash(initial)\n                detail["exact_call_sequence_match"] = prediction["tool_calls"] == row["label"]["tool_calls"]\n            detail["seconds"] = time.monotonic() - tick\n            submissions.write(json.dumps(prediction, ensure_ascii=False) + "\\n")\n            traces.write(json.dumps(detail, ensure_ascii=False) + "\\n")\n            submissions.flush()\n            traces.flush()\n            summaries.append({k: v for k, v in detail.items() if k != "trace"})\n            print(json.dumps({"completed": i, "total": len(rows), **summaries[-1]}), flush=True)\n    from collections import Counter\n    report = {"samples": len(rows), "mode": a.mode, "seconds": time.monotonic()-started,\n              "statuses": dict(Counter(r["status"] for r in summaries)),\n              "note": "Local diagnostics, not the official fuzzy call F1/composite score.",\n              "generation": generator.stats,\n              "peak_cuda_gib": generator.torch.cuda.max_memory_allocated()/2**30}\n    if a.split == "train":\n        for metric in ("canonical_state_match", "exact_call_sequence_match"):\n            report[metric] = sum(r[metric] for r in summaries)/len(rows)\n    (a.output/"report.json").write_text(json.dumps(report, indent=2))\n\nif __name__ == "__main__":\n    main()\n', 'transcribe.py': '"""Transcribe selected current AND history user audio with supplied Zipformer."""\nimport argparse\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path\nfrom run import read_rows\n\ndef main():\n    p = argparse.ArgumentParser(description=__doc__)\n    p.add_argument("--data-root", type=Path, required=True)\n    p.add_argument("--split", default="train", choices=["train", "public_test"])\n    p.add_argument("--ids", type=Path)\n    p.add_argument("--limit", type=int)\n    p.add_argument("--model-dir", type=Path, required=True)\n    p.add_argument("--output", type=Path, required=True)\n    a = p.parse_args()\n    import numpy as np\n    import soundfile as sf\n    import sherpa_onnx\n    from scipy.signal import resample_poly\n    rows = read_rows(a.data_root, a.split, a.ids, a.limit)\n    names = list(dict.fromkeys(\n        name for row in rows for name in\n        [row["audio"]] + [m["audio"] for m in row["history"] if m["role"] == "user"]))\n    if not names:\n        raise ValueError("No audio selected")\n    paths = {key: a.model_dir/f"{key}.onnx" for key in ("encoder", "decoder", "joiner")}\n    paths["tokens"] = a.model_dir/"tokens.txt"\n    hashes = {key: hashlib.sha256(path.read_bytes()).hexdigest() for key, path in paths.items()}\n    recognizer = sherpa_onnx.OfflineRecognizer.from_transducer(\n        **{key: str(path) for key, path in paths.items()}, num_threads=4,\n        sample_rate=16000, feature_dim=80, decoding_method="greedy_search", provider="cpu")\n    if a.output.exists():\n        raise ValueError("Transcript output already exists; choose a fresh path")\n    a.output.parent.mkdir(parents=True, exist_ok=True)\n    texts = {}\n    for index, name in enumerate(names, 1):\n        base = (a.data_root/a.split).resolve()\n        path = (base/name).resolve()\n        if not path.is_relative_to(base):\n            raise ValueError("Audio path escapes split root")\n        samples, rate = sf.read(path, dtype="float32", always_2d=True)\n        samples = samples.mean(axis=1)\n        if not len(samples) or not np.isfinite(samples).all():\n            raise ValueError(f"Invalid audio: {name}")\n        if rate != 16000:\n            factor = math.gcd(rate, 16000)\n            samples = resample_poly(samples, 16000//factor, rate//factor)\n        stream = recognizer.create_stream()\n        stream.accept_waveform(16000, np.ascontiguousarray(samples, dtype=np.float32))\n        recognizer.decode_stream(stream)\n        texts[name] = stream.result.text.strip()\n        if not texts[name]:\n            raise ValueError(f"Empty ASR result: {name}")\n        print(f"ASR {index}/{len(names)} {name}", flush=True)\n    a.output.write_text(json.dumps(texts, ensure_ascii=False, indent=2))\n    a.output.with_suffix(".meta.json").write_text(json.dumps(\n        {"split": a.split, "selected_ids": [r["id"] for r in rows],\n         "model_sha256": hashes, "sherpa_onnx": sherpa_onnx.__version__,\n         "provider": "cpu", "current_and_history": True}, indent=2))\n\nif __name__ == "__main__":\n    main()\n', 'test_agent.py': 'import copy\nimport json\nimport unittest\nfrom agent import messages_for, parse_action, run_episode\n\nCATALOG = {"apps": [{"tools": [\n    {"name": "contacts.search", "description": "Find", "parameters": {\n        "type": "object", "properties": {"query": {"type": "string"}}}},\n    {"name": "sms.send", "description": "Send", "parameters": {\n        "type": "object", "properties": {"contact_id": {"type": "string"},\n                                        "content": {"type": "string"}}}},\n    {"name": "sys.ask_missing_info", "description": "Ask", "parameters": {\n        "type": "object", "properties": {"tool_name": {"type": "string"},\n                                        "params": {"type": "array"}}}}\n]}]}\nITEM = {"id": "synthetic", "audio": "now.wav", "audio_text": "Nhắn Nam xin chào",\n        "history": [], "context": {"installed_apps": {\n            "contacts": {"tools": ["contacts.search"]},\n            "sms": {"tools": ["sms.send"]}}}}\nTOOLS = {t["name"]: t for t in CATALOG["apps"][0]["tools"]}\n\ndef xml(name, args):\n    return "<tool_call>\\n<function=" + name + ">\\n" + "".join(\n        "<parameter=" + k + ">\\n" +\n        (v if isinstance(v, str) else json.dumps(v)) + "\\n</parameter>\\n"\n        for k, v in args.items()) + "</function>\\n</tool_call>"\n\nclass Env:\n    def __init__(self):\n        self.calls = []\n    def validate(self, name, args):\n        return {"ok": not (name == "sms.send" and "contact_id" not in args)}\n    def call(self, name, args):\n        self.calls.append({"name": name, "arguments": args})\n        return {"ok": True, "contacts": [{"id": "observed_42"}]}\n    def get_tool_calls(self):\n        return copy.deepcopy(self.calls)\n    def get_state_hash(self):\n        return "hash"\n\nclass AgentTests(unittest.TestCase):\n    def test_observation_drives_next_action(self):\n        env = Env()\n        def generate(messages, tools):\n            self.assertNotIn("label", str(messages))\n            if not env.calls:\n                return xml("contacts.search", {"query": "Nam"})\n            if len(env.calls) == 1:\n                result = json.loads(messages[-1]["content"])\n                return xml("sms.send", {"contact_id": result["contacts"][0]["id"],\n                                        "content": "xin chào"})\n            return "Đã gửi."\n        pred, detail = run_episode(ITEM, CATALOG, env, generate)\n        self.assertEqual(pred["tool_calls"][1]["arguments"]["contact_id"], "observed_42")\n        self.assertEqual(detail["status"], "stopped")\n\n    def test_history_is_evidence_not_replayed(self):\n        item = copy.deepcopy(ITEM)\n        item["history"] = [{"role": "user", "audio": "old.wav", "audio_text": "Tìm Nam"},\n            {"role": "assistant", "text": "Đã tìm.", "tool_calls": [\n                {"name": "contacts.search", "arguments": {"query": "Nam"}}],\n             "tool_results": [{"ok": True, "contacts": [{"id": "old_id"}]}]}]\n        env = Env()\n        pred, _ = run_episode(item, CATALOG, env, lambda m, t: "Không cần thao tác.")\n        self.assertEqual(pred["tool_calls"], [])\n        self.assertIn("old_id", str(messages_for(item)))\n\n    def test_asr_uses_history_and_current_without_oracle_fallback(self):\n        item = copy.deepcopy(ITEM)\n        item["history"] = [{"role": "user", "audio": "old.wav", "audio_text": "ORACLE"}]\n        messages = messages_for(item, "asr", {"old.wav": "ASR old", "now.wav": "ASR now"})\n        self.assertEqual([m["content"] for m in messages[1:]], ["ASR old", "ASR now"])\n        with self.assertRaises(KeyError):\n            messages_for(item, "asr", {"now.wav": "now"})\n\n    def test_schema_rejection_does_not_execute(self):\n        env = Env()\n        pred, detail = run_episode(ITEM, CATALOG, env, lambda m, t: xml(\n            "sms.send", {"content": "hi"}))\n        self.assertEqual(pred["tool_calls"], [])\n        self.assertEqual(detail["status"], "error")\n\n    def test_missing_info_is_executed_and_terminal(self):\n        env = Env()\n        pred, detail = run_episode(ITEM, CATALOG, env, lambda m, t: xml(\n            "sys.ask_missing_info", {"tool_name": "sms.send", "params": ["contact_id"]}))\n        self.assertEqual(len(pred["tool_calls"]), 1)\n        self.assertEqual(detail["status"], "missing_info")\n\n    def test_reject_malformed_multiple_unknown_calls(self):\n        call = xml("contacts.search", {"query": "Nam"})\n        for raw in [call + call, call[:-4], call + "suffix",\n                    xml("unknown.tool", {}), "<tool_call>broken</tool_call>"]:\n            with self.subTest(raw=raw), self.assertRaises(ValueError):\n                parse_action(raw, TOOLS)\n\n    def test_step_budget_preserves_executed_calls(self):\n        env = Env()\n        pred, detail = run_episode(ITEM, CATALOG, env, lambda m, t: xml(\n            "contacts.search", {"query": "Nam"}), max_steps=2)\n        self.assertEqual(len(pred["tool_calls"]), 2)\n        self.assertEqual(detail["status"], "step_limit")\n\nif __name__ == "__main__":\n    unittest.main()\n', 'requirements.txt': '# Separate environment from the Qwen3 / Transformers 4.x training notebooks.\ntransformers>=5.3,<6\naccelerate>=1.12,<2\ntorch>=2.6\nsafetensors>=0.4.5\nsherpa-onnx>=1.12,<2\nsoundfile>=0.13,<1\nscipy>=1.13,<2\nnumpy>=1.26,<3\n', 'validation_ids.txt': 'train_0001\ntrain_0004\ntrain_0015\ntrain_0016\ntrain_0017\ntrain_0031\ntrain_0063\ntrain_0066\ntrain_0075\ntrain_0080\ntrain_0088\ntrain_0090\ntrain_0095\ntrain_0106\ntrain_0110\ntrain_0117\ntrain_0130\ntrain_0133\ntrain_0141\ntrain_0144\ntrain_0145\ntrain_0147\ntrain_0162\ntrain_0163\ntrain_0164\ntrain_0169\ntrain_0172\ntrain_0181\ntrain_0194\ntrain_0196\ntrain_0203\ntrain_0207\ntrain_0214\ntrain_0216\ntrain_0219\ntrain_0225\ntrain_0228\ntrain_0231\ntrain_0233\ntrain_0236\ntrain_0240\ntrain_0255\ntrain_0278\ntrain_0285\ntrain_0304\ntrain_0313\ntrain_0316\ntrain_0321\ntrain_0330\ntrain_0344\ntrain_0345\ntrain_0346\ntrain_0350\ntrain_0354\ntrain_0357\ntrain_0378\ntrain_0381\ntrain_0395\ntrain_0405\ntrain_0422\ntrain_0435\ntrain_0442\ntrain_0453\ntrain_0457\ntrain_0468\ntrain_0469\ntrain_0476\ntrain_0480\ntrain_0481\ntrain_0486\ntrain_0497\ntrain_0500\ntrain_0505\ntrain_0508\ntrain_0518\ntrain_0536\ntrain_0538\ntrain_0550\ntrain_0554\ntrain_0560\ntrain_0563\ntrain_0572\ntrain_0577\ntrain_0596\ntrain_0598\ntrain_0618\ntrain_0621\ntrain_0624\ntrain_0625\ntrain_0628\ntrain_0664\ntrain_0689\ntrain_0694\ntrain_0719\ntrain_0729\ntrain_0752\ntrain_0764\ntrain_0770\ntrain_0772\ntrain_0777\ntrain_0779\ntrain_0787\ntrain_0790\ntrain_0800\ntrain_0805\ntrain_0814\ntrain_0815\ntrain_0819\ntrain_0820\ntrain_0834\ntrain_0857\ntrain_0864\ntrain_0867\ntrain_0870\ntrain_0875\ntrain_0923\ntrain_0933\ntrain_0935\ntrain_0943\ntrain_0945\ntrain_0947\ntrain_0948\ntrain_0953\ntrain_0958\ntrain_0959\ntrain_0960\ntrain_0977\ntrain_0978\ntrain_0991\ntrain_1000\ntrain_1014\ntrain_1016\ntrain_1019\ntrain_1021\ntrain_1049\ntrain_1060\ntrain_1062\ntrain_1070\ntrain_1081\ntrain_1089\ntrain_1091\ntrain_1096\ntrain_1097\ntrain_1109\ntrain_1117\ntrain_1118\ntrain_1123\ntrain_1124\ntrain_1128\ntrain_1137\ntrain_1147\ntrain_1148\ntrain_1155\ntrain_1160\ntrain_1163\ntrain_1175\ntrain_1180\ntrain_1188\ntrain_1201\ntrain_1206\ntrain_1209\ntrain_1215\ntrain_1216\ntrain_1233\ntrain_1234\ntrain_1246\ntrain_1248\ntrain_1249\ntrain_1253\ntrain_1265\ntrain_1278\ntrain_1289\ntrain_1295\ntrain_1300\ntrain_1305\ntrain_1310\ntrain_1311\ntrain_1317\ntrain_1325\ntrain_1346\ntrain_1351\ntrain_1352\ntrain_1353\ntrain_1355\ntrain_1356\ntrain_1363\ntrain_1373\ntrain_1379\ntrain_1380\ntrain_1384\ntrain_1399\ntrain_1405\ntrain_1408\ntrain_1432\ntrain_1433\ntrain_1437\ntrain_1440\ntrain_1448\ntrain_1458\ntrain_1460\ntrain_1471\ntrain_1472\ntrain_1474\ntrain_1475\ntrain_1476\ntrain_1478\ntrain_1486\ntrain_1491\ntrain_1497\ntrain_1506\ntrain_1512\ntrain_1519\ntrain_1526\ntrain_1528\ntrain_1533\ntrain_1540\ntrain_1542\ntrain_1544\ntrain_1546\ntrain_1556\ntrain_1557\ntrain_1560\ntrain_1571\ntrain_1572\ntrain_1589\ntrain_1598\ntrain_1606\ntrain_1608\ntrain_1615\ntrain_1617\ntrain_1623\ntrain_1653\ntrain_1654\ntrain_1658\ntrain_1661\ntrain_1668\ntrain_1697\ntrain_1702\ntrain_1703\ntrain_1705\ntrain_1707\ntrain_1712\ntrain_1713\ntrain_1715\ntrain_1746\n'}
for name, content in PAYLOAD.items():
    (WORK/name).write_text(content)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                str(WORK/"requirements.txt")], check=True)
subprocess.run([sys.executable, "-B", str(WORK/"test_agent.py")], check=True)


In [ ]:
assert PROFILE in {"smoke", "validation", "public"}
split = "public_test" if PROFILE == "public" else "train"
if split == "public_test":
    assert MODE == "asr", "Set MODE=asr for public test"
assert MODE in {"oracle", "asr"}
input_root = Path("/kaggle/input")
if DATA_ROOT:
    data_root = Path(DATA_ROOT)
else:
    candidates = sorted({p.parent.parent for p in input_root.rglob(f"{split}.jsonl")
                         if p.parent.name == split})
    if not candidates:
        archives = [Path(DATA_ZIP)] if DATA_ZIP else sorted(input_root.rglob("train+public_test.zip"))
        assert len(archives) == 1, "Set DATA_ROOT or DATA_ZIP explicitly"
        extracted = Path("/kaggle/working/lifemate_data")
        assert not extracted.exists(), "Extraction already exists; set DATA_ROOT to it"
        extracted.mkdir()
        with zipfile.ZipFile(archives[0]) as archive:
            for member in archive.infolist():
                target = (extracted/member.filename).resolve()
                assert target.is_relative_to(extracted.resolve()), "Unsafe ZIP path"
            archive.extractall(extracted)
        candidates = sorted({p.parent.parent for p in extracted.rglob(f"{split}.jsonl")
                             if p.parent.name == split})
    assert len(candidates) == 1, f"Ambiguous dataset roots: {candidates}"
    data_root = candidates[0]
assert (data_root/split/"zaic_tools"/"__init__.py").is_file()
output = Path("/kaggle/working")/RUN_NAME
assert not output.exists(), "Change RUN_NAME for a new run"
selection = ["--ids", str(WORK/"validation_ids.txt")] if PROFILE == "validation" else []
if PROFILE == "smoke":
    selection += ["--limit", "8"]
common = ["--data-root", str(data_root), "--split", split] + selection
print("DATA_ROOT:", data_root, "PROFILE:", PROFILE, "MODE:", MODE)


In [ ]:
transcript_args = []
if MODE == "asr":
    assert ASR_MODEL_DIR, "Set ASR_MODEL_DIR to the supplied Zipformer ONNX directory"
    transcript_path = Path("/kaggle/working")/(RUN_NAME + "_transcripts.json")
    subprocess.run([sys.executable, "-B", str(WORK/"transcribe.py"), *common,
                    "--model-dir", ASR_MODEL_DIR, "--output", str(transcript_path)], check=True)
    transcript_args = ["--transcripts", str(transcript_path)]
command = [sys.executable, "-B", str(WORK/"run.py"), *common,
           "--mode", MODE, *transcript_args, "--model", MODEL, "--revision", REVISION,
           "--max-input", str(MAX_INPUT), "--max-new-tokens", str(MAX_NEW_TOKENS),
           "--max-steps", str(MAX_STEPS), "--output", str(output)]
subprocess.run(command, check=True)


In [ ]:
print((output/"report.json").read_text())
# Schema validation of actual submission; labels are never passed to inference.
env = dict(os.environ, PYTHONPATH=str(data_root/split))
subprocess.run([sys.executable, "-m", "zaic_tools", "--split", split,
                "--root", str(data_root), "--validate", str(output/"submission.jsonl")],
               env=env, check=True)
import shutil
archive = shutil.make_archive(str(output), "zip", output)
print("Artifacts:", archive)
from IPython.display import FileLink, display
display(FileLink(str(Path(archive).relative_to("/kaggle/working"))))
